# 🚀 ComfyUI + MiniMax-H3 on Google Colab (A100 GPU Edition)

Google AI Pro 等のプランで付与される **Colab Compute Units (CU)** を活用して、強力な最新動画生成モデル **MiniMax-H3 (Hailuo)** を A100 (40GB VRAM) 環境で動かすためのオールインワン検証テンプレートです。

### 💡 特徴・ポイント
- **A100 GPU 最適化**: MiniMax-H3 の大規模モデル (`int8_convrot` 等) + Text Encoder + Audio/Video VAE をストレスなくロード。
- **ComfyUI 公式最新サポート**: v0.3.0 以降のネイティブ MiniMax-H3 ノード対応。
- **Cloudflare Tunnel (無料・トークン不要)**: ngrok 不要ですぐにセキュアな一時公開 URL (`trycloudflare.com`) を自動発行。
- **Google Drive 連携オプション**: モデルを毎回落とさず Google Drive にキャッシュ可能。

> ⚠️ **注意**: ノートブック上部のメニュー「ランタイム」→「ランタイムのタイプを変更」から、**GPU (A100)** が選択されていることを確認してください。

## Step 1: 環境確認 & Google Drive マウント (任意)

In [ ]:
# GPU の確認 (A100 がアサインされているか確認)
!nvidia-smi

# (任意) Google Drive をマウントしてモデルを永続化したい場合は以下を実行
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## Step 2: ComfyUI 本体 & 必須カスタムノードのセットアップ

最新の ComfyUI および ComfyUI-Manager をセットアップします。

In [ ]:
import os

# 作業ディレクトリへ移動
%cd /content

# ComfyUI 本体のクローン (最新版)
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    %cd /content/ComfyUI
    !git pull
    %cd /content

# 依存ライブラリのインストール
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q sageattention huggingface_hub

# ComfyUI-Manager のインストール
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# MiniMax-H3 Easy / 補助ノード (任意で便利に利用可能)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy"):
    !git clone https://github.com/kijai/ComfyUI-MiniMaxH3-Easy.git || true

%cd /content/ComfyUI

## Step 3: MiniMax-H3 モデルのダウンロード

MiniMax-H3 に必要なコンポーネント (Hugging Face `Comfy-Org/MiniMax-H3`) をダウンロードします。
- **Diffusion Model**: `minimax_h3_fl2va_pruned_int8_convrot.safetensors` (推奨・高効率版)
- **Text Encoder**: `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`
- **VAE**: `minimax_h3_video_vae_fp16.safetensors` & `minimax_h3_audio_vae_fp32.safetensors`

In [ ]:
import os
from huggingface_hub import hf_hub_download

REPO_ID = "Comfy-Org/MiniMax-H3"

# モデル格納先ディレクトリ
DIFFUSION_DIR = "/content/ComfyUI/models/diffusion_models"
TEXT_ENCODER_DIR = "/content/ComfyUI/models/text_encoders"
VAE_DIR = "/content/ComfyUI/models/vae"

os.makedirs(DIFFUSION_DIR, exist_ok=True)
os.makedirs(TEXT_ENCODER_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)

print("📥 MiniMax-H3 のモデルファイルをダウンロード中 (初回は約数分かかります)...")

# 1. Diffusion Model (約15~20GB)
hf_hub_download(
    repo_id=REPO_ID,
    filename="diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    local_dir="/content/ComfyUI/models",
    local_dir_use_symlinks=False
)

# 2. Text Encoder (Qwen3VL)
hf_hub_download(
    repo_id=REPO_ID,
    filename="text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    local_dir="/content/ComfyUI/models",
    local_dir_use_symlinks=False
)

# 3. Video & Audio VAE
hf_hub_download(
    repo_id=REPO_ID,
    filename="vae/minimax_h3_video_vae_fp16.safetensors",
    local_dir="/content/ComfyUI/models",
    local_dir_use_symlinks=False
)
hf_hub_download(
    repo_id=REPO_ID,
    filename="vae/minimax_h3_audio_vae_fp32.safetensors",
    local_dir="/content/ComfyUI/models",
    local_dir_use_symlinks=False
)

print("✅ すべての MiniMax-H3 モデルの準備が完了しました！")

## Step 4: ComfyUI 起動 & Cloudflare Tunnel 経由でアクセス

バックグラウンドで ComfyUI を起動し、Cloudflare Tunnel (`trycloudflare.com`) の安全なパブリック URL を発行します。
表示された `https://xxxx.trycloudflare.com` のリンクをクリックすると ComfyUI の WebUI が開きます。

### 💡 MiniMax-H3 ワークフローの読み込み方
1. ComfyUI を開いたら、画面メニューの **[Templates]** (またはワークフロー一覧) を開きます。
2. **[Video]** カテゴリ内にある **MiniMax H3 Text-to-Video** または **Image-to-Video** テンプレートを選択します。
3. ダウンロードした各モデルが自動認識されますので、プロンプトを入力して **[Queue]** を実行します。

In [ ]:
import subprocess
import threading
import time
import re

# Cloudflared のダウンロード
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# ComfyUI をバックグラウンドで起動
print("⚡ Starting ComfyUI...")
%cd /content/ComfyUI
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# ComfyUI の起動ログを別スレッドで監視
def log_comfy():
    for line in iter(comfy_proc.stdout.readline, ''):
        print("[ComfyUI]", line, end="")

threading.Thread(target=log_comfy, daemon=True).start()

# 少し待機して Cloudflared トンネルを起動
time.sleep(5)
print("🌐 Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 ComfyUI is LIVE: {url}")
        print("="*60 + "\n")
        break